# MS MARCO RARS-v5 PQ-Aware 100K Pilot

## Goal

Run the frozen, development-only rank-aware hard-PQ adapter gate at implementation commit `93105ae1895974e28b34952c2dd777f037c6e0bf`. The experiment asks whether two rank-8 residual adapters can recover end-to-end known-positive Recall@100 on a deterministic 100K corpus without harming Recall@10 or FP32 retrieval.

This is not official MS MARCO Recall, independent confirmation, a new-QAT claim, a frozen-index retrofit, or evidence of SIGIR readiness.

## Method and constraints

The BGE encoders, IVF centroids, IVF list assignments, and M32 PQ codebooks remain fixed. Hard residual-PQ assignments are recomputed for adapted documents; an identity straight-through estimator supplies gradients. Checkpoint selection uses fresh end-to-end 100K IVF-PQ retrieval with `nprobe=16`, never soft-PQ or label-injected selection candidates.

Training uses the already-observed v3 design role; epoch selection uses the already-observed v3 diagnostic-audit role. Zero labels are unjudged mined negatives, not explicit non-relevant judgments. The 803-query future role remains identity-only and is never opened for candidates, labels, or metrics.

In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

EXPERIMENT_PYTHON = sys.executable
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q',
    'faiss-gpu-cu12==1.12.0', 'pytest>=8,<9',
], check=True)
NUMPY_TARGET = Path('/content/rars-v5-numpy126')
if NUMPY_TARGET.exists():
    shutil.rmtree(NUMPY_TARGET)
NUMPY_TARGET.mkdir(parents=True)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pip', 'install', '-q', '--no-deps',
    '--target', str(NUMPY_TARGET), 'numpy==1.26.4',
], check=True)
EXPERIMENT_ENV = os.environ.copy()
EXPERIMENT_ENV['PYTHONPATH'] = os.pathsep.join(filter(None, [
    str(NUMPY_TARGET), EXPERIMENT_ENV.get('PYTHONPATH', ''),
]))
EXPERIMENT_ENV['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
installed_numpy_version = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c', 'import numpy; print(numpy.__version__)',
], text=True, env=EXPERIMENT_ENV).strip()
assert installed_numpy_version == '1.26.4', installed_numpy_version
installed_numpy_path = subprocess.check_output([
    EXPERIMENT_PYTHON, '-c', 'import numpy; print(numpy.__file__)',
], text=True, env=EXPERIMENT_ENV).strip()
assert Path(installed_numpy_path).resolve().is_relative_to(NUMPY_TARGET.resolve())
print('Colab host-kernel NumPy (not used by experiments):',
      getattr(sys.modules.get('numpy'), '__version__', 'not-loaded'))
print('Fresh experiment-subprocess NumPy:', installed_numpy_version)

from google.colab import drive
drive.mount('/content/drive')

import hashlib, json

TRAINING_COMMIT = 'bb9b106e69b9a453756fd800665f701614ce67b3'
V3_IMPLEMENTATION_COMMIT = '05c2ae43b7d11783460822d10c590240dab1a399'
V5_IMPLEMENTATION_COMMIT = '93105ae1895974e28b34952c2dd777f037c6e0bf'
REPO_URL = 'https://github.com/ravan-chuang/Embedding_Compression_for_RAG_Retrieval.git'
TRAIN_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v2_2')
# Preserve the exact clone path registered by the completed v3 design freeze.
V3_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v3_oracle')
V5_REPO = Path('/content/Embedding_Compression_for_RAG_Retrieval_rars_v5')

PARENT_WORK = Path('/content') / f'rars-v2.2-{TRAINING_COMMIT[:12]}'
V3_WORK = Path('/content') / f'rars-v3-{V3_IMPLEMENTATION_COMMIT[:12]}'
V5_WORK = Path('/content') / f'rars-v5-{V5_IMPLEMENTATION_COMMIT[:12]}'
for local_work in (PARENT_WORK, V3_WORK, V5_WORK):
    if local_work.exists():
        shutil.rmtree(local_work)
    local_work.mkdir(parents=True)
PARENT_BUNDLES = PARENT_WORK / 'bundles'
PARENT_CANDIDATE_CACHE = PARENT_WORK / 'candidate-cache'
V3_BUNDLES = V3_WORK / 'bundles'
V5_BUNDLE = V5_WORK / 'pilot-bundle'

DRIVE = Path('/content/drive/MyDrive/rag-pq-checkpoints')
CACHE = DRIVE / 'msmarco_basis_gate0_cache'
CLEAN = DRIVE / 'rars_clean_split_v1'
PCA = DRIVE / 'rars_pca_comparator_v1'
INDEX = DRIVE / 'msmarco_1m_pq_residual_gate3/frozen_ivfpq_m32_nlist512.index'
V3_OUTPUT = DRIVE / 'rars-v3-oracle-first' / V3_IMPLEMENTATION_COMMIT[:12]
OUTPUT = DRIVE / 'rars-v5-pq-aware-100k' / V5_IMPLEMENTATION_COMMIT[:12] / 'seed42'

def sha256_file(path, chunk_size=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b''):
            digest.update(chunk)
    return digest.hexdigest()

def verify_record(path, record):
    path = Path(path)
    assert path.is_file(), path
    assert path.stat().st_size == int(record['bytes']), path
    assert sha256_file(path) == record['sha256'], path

EXPERIMENT_PROBE = r'''
import json, os, sys
import faiss, numpy as np, torch
print(json.dumps({
    'python_version': '.'.join(map(str, sys.version_info[:3])),
    'python_full': sys.version,
    'numpy_version': np.__version__,
    'numpy_module_path': np.__file__,
    'torch_version': torch.__version__,
    'torch_cuda_version': str(torch.version.cuda),
    'cuda_available': torch.cuda.is_available(),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'faiss_version': str(getattr(faiss, '__version__', 'UNKNOWN')),
    'cublas_workspace_config': os.environ.get('CUBLAS_WORKSPACE_CONFIG'),
}, allow_nan=False))
'''

def probe_experiment_environment():
    return json.loads(subprocess.check_output(
        [EXPERIMENT_PYTHON, '-c', EXPERIMENT_PROBE],
        text=True, env=EXPERIMENT_ENV,
    ))

In [ ]:
def clone_exact(destination, commit):
    if destination.exists():
        shutil.rmtree(destination)
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(destination)], check=True)
    subprocess.run(['git', '-C', str(destination), 'checkout', '--detach', commit], check=True)
    head = subprocess.check_output(
        ['git', '-C', str(destination), 'rev-parse', 'HEAD'], text=True
    ).strip()
    dirty = subprocess.check_output(
        ['git', '-C', str(destination), 'status', '--porcelain'], text=True
    ).strip()
    assert head == commit, (head, commit)
    assert not dirty, dirty

clone_exact(TRAIN_REPO, TRAINING_COMMIT)
clone_exact(V3_REPO, V3_IMPLEMENTATION_COMMIT)
clone_exact(V5_REPO, V5_IMPLEMENTATION_COMMIT)

V3_PROTOCOL_PATH = V3_REPO / 'protocols/rars_v3_oracle_first_feasibility_v1.json'
V5_PROTOCOL_PATH = V5_REPO / 'protocols/rars_v5_pq_aware_100k_pilot_v1.json'
v3_protocol = json.loads(V3_PROTOCOL_PATH.read_text())
protocol = json.loads(V5_PROTOCOL_PATH.read_text())
assert protocol['status'] == 'IMPLEMENTATION_FROZEN_BEFORE_FIRST_100K_PILOT_OUTCOME'
assert protocol['method_revision_allowed'] is False
assert protocol['outcome_informed_revision_allowed'] is False
assert protocol['data_policy']['future_method_holdout']['allowed_to_open'] is False
assert protocol['parent_lineage']['v3_implementation_commit'] == V3_IMPLEMENTATION_COMMIT

subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v2_2_core.py',
    'tests/test_freeze_rars_v2_2_inner_bundles.py',
    'tests/test_build_msmarco_rars_v2_boundary_bundles.py',
], cwd=TRAIN_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v3_oracle_core.py',
    'tests/test_build_msmarco_rars_v3_oracle_bundles.py',
    'tests/test_materialize_rars_v3_role_labels.py',
    'tests/test_rars_v3_oracle_protocol_contract.py',
], cwd=V3_REPO, check=True, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, '-m', 'pytest', '-q',
    'tests/test_rars_v5_pq_aware_core.py',
    'tests/test_build_rars_v5_pq_aware_100k_bundle.py',
    'tests/test_rars_v5_pq_aware_protocol_contract.py',
], cwd=V5_REPO, check=True, env=EXPERIMENT_ENV)
print('Exact v5 implementation commit:', V5_IMPLEMENTATION_COMMIT)
print('Frozen v5 protocol SHA-256:', sha256_file(V5_PROTOCOL_PATH))

In [ ]:
required = [
    CACHE / 'embeddings.fp16.memmap',
    CACHE / 'doc_ids.int64.memmap',
    CACHE / 'query_vectors.fp32.npy',
    CACHE / 'qrels_subset.json',
    INDEX,
    PCA / 'bases/pca_unweighted_rank16.float32.npy',
    PCA / 'sidecars/scales_pca_rank16.float32.npy',
    PCA / 'sidecars/codes_pca_rank16.int8.memmap',
    CLEAN / 'selected_config.json',
    CLEAN / 'bases/score_error_weighted_rank16.npy',
    CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy',
    CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap',
    V3_OUTPUT / 'oracle_complete.json',
    V3_OUTPUT / 'oracle_summary.json',
    V3_OUTPUT / 'design_freeze.json',
]
missing = [str(path) for path in required if not path.exists()]
assert not missing, {'missing_artifacts': missing}
assert shutil.disk_usage('/content').free >= 6_000_000_000, 'Need 6 GB local disk'
assert not OUTPUT.exists() or not any(OUTPUT.iterdir()), (
    'The durable seed-42 output is non-empty. Do not delete or overwrite it; '
    'return its contents for audit before deciding how to continue.'
)

current_environment = probe_experiment_environment()
contract = protocol['execution_environment_contract']
assert current_environment['python_version'] == contract['python_version']
assert current_environment['numpy_version'] == contract['numpy_version']
assert Path(current_environment['numpy_module_path']).resolve().is_relative_to(
    NUMPY_TARGET.resolve()
)
assert current_environment['torch_version'] == contract['torch_version']
assert current_environment['torch_cuda_version'] == contract['torch_cuda_version']
assert current_environment['cuda_available'] is True
assert contract['gpu_name_must_contain'] in current_environment['gpu_name']
assert current_environment['faiss_version'] != 'UNKNOWN'
assert current_environment['cublas_workspace_config'] == contract['cublas_workspace_config']
v3_summary = json.loads((V3_OUTPUT / 'oracle_summary.json').read_text())
assert v3_summary['formal_decision'] == protocol['parent_lineage']['v3_observed_formal_decision']
print(json.dumps(current_environment, indent=2))

## Reproduce the registered development inputs

The next cells rematerialize the exact v2.2 parent and v3 candidate roles at their pinned commits. The inherited v2.2 builder necessarily reads the historical qrels cache. The v5 builder itself receives no qrels path. It reads only the already-observed design/audit role labels.

The audit-label release is validated against the previously completed v3 design freeze. The future role is checked only by filename and must remain identity-only.

In [ ]:
builder = [
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/build_msmarco_rars_v2_boundary_bundles.py'),
    '--inner-only',
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--index', str(INDEX),
    '--qrels', str(CACHE / 'qrels_subset.json'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--cache-root', str(PARENT_CANDIDATE_CACHE),
    '--pca-config', str(TRAIN_REPO / 'results/rars_pca_comparator/selected_pca_config.json'),
    '--pca-basis', str(PCA / 'bases/pca_unweighted_rank16.float32.npy'),
    '--pca-scales', str(PCA / 'sidecars/scales_pca_rank16.float32.npy'),
    '--pca-codes', str(PCA / 'sidecars/codes_pca_rank16.int8.memmap'),
    '--rars-config', str(CLEAN / 'selected_config.json'),
    '--rars-basis', str(CLEAN / 'bases/score_error_weighted_rank16.npy'),
    '--rars-scales', str(CLEAN / 'sidecars/scales_score_error_weighted_rank16.float32.npy'),
    '--rars-codes', str(CLEAN / 'sidecars/codes_score_error_weighted_rank16.int8.memmap'),
    '--output-root', str(PARENT_BUNDLES),
    '--residual-batch-size', '20000',
]
subprocess.run(builder, check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)
bundle_summary = json.loads((PARENT_BUNDLES / 'bundle_build_summary.json').read_text())
assert bundle_summary['outer_validation_built'] is False
assert set(bundle_summary['roles']) == {'inner_train', 'inner_validation'}

subprocess.run([
    EXPERIMENT_PYTHON, str(TRAIN_REPO / 'scripts/freeze_rars_v2_2_inner_bundles.py'),
    '--bundle-root', str(PARENT_BUNDLES),
    '--query-vectors', str(CACHE / 'query_vectors.fp32.npy'),
    '--train-split', str(TRAIN_REPO / 'splits/msmarco_rars_train_split.json'),
    '--outer-validation-split', str(TRAIN_REPO / 'splits/msmarco_rars_validation_split.json'),
    '--clean-test-split', str(TRAIN_REPO / 'splits/msmarco_rars_test_split.json'),
    '--source-commit', TRAINING_COMMIT,
], check=True, cwd=TRAIN_REPO, env=EXPERIMENT_ENV)

parent = v3_protocol['parent_lineage']
parent_hashes = {
    'parent_inner_train_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_train/v2_2_manifest.json'),
    'parent_inner_train_source_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_train/manifest.json'),
    'parent_inner_train_query_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_train/query_manifest.json'),
    'closed_inner_validation_query_manifest_sha256': sha256_file(PARENT_BUNDLES / 'inner_validation/query_manifest.json'),
    'parent_v2_2_split_audit_sha256': sha256_file(PARENT_BUNDLES / 'v2_2_split_audit.json'),
}
for key, actual in parent_hashes.items():
    assert actual == parent[key], (key, actual, parent[key])
print('Exact v2.2 parent rematerialized.')

In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON, str(V3_REPO / 'scripts/build_msmarco_rars_v3_oracle_bundles.py'),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--doc-ids', str(CACHE / 'doc_ids.int64.memmap'),
    '--output-root', str(V3_BUNDLES),
    '--protocol', str(V3_PROTOCOL_PATH),
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--n-docs', '1000000',
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)
candidate_summary = json.loads(
    (V3_BUNDLES / 'v3_oracle_bundle_freeze_summary.json').read_text()
)
assert candidate_summary['status'] == 'V3_QRELS_FREE_CANDIDATE_BUNDLES_FROZEN'
assert candidate_summary['parent_label_payload_bytes_read'] is False
assert candidate_summary['qrels_opened_or_parsed'] is False

ROLE_LABEL_FILES = (
    'candidate_relevance.uint8.npy',
    'relevant_counts.int32.npy',
    'v3_role_labels_started.json',
    'v3_role_labels_manifest.json',
)
future_files = {path.name for path in (V3_BUNDLES / 'future_method_holdout').iterdir()}
assert future_files == {'query_manifest.json', 'v3_identity_manifest.json'}

subprocess.run([
    EXPERIMENT_PYTHON, str(V3_REPO / 'scripts/materialize_rars_v3_role_labels.py'),
    '--bundle-root', str(V3_BUNDLES),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--role', 'oracle_design',
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--protocol', str(V3_PROTOCOL_PATH),
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)
subprocess.run([
    EXPERIMENT_PYTHON, str(V3_REPO / 'scripts/materialize_rars_v3_role_labels.py'),
    '--bundle-root', str(V3_BUNDLES),
    '--parent-inner-train-bundle', str(PARENT_BUNDLES / 'inner_train'),
    '--role', 'oracle_audit',
    '--source-commit', V3_IMPLEMENTATION_COMMIT,
    '--protocol', str(V3_PROTOCOL_PATH),
    '--design-freeze', str(V3_OUTPUT / 'design_freeze.json'),
], check=True, cwd=V3_REPO, env=EXPERIMENT_ENV)
for role in ('oracle_design', 'oracle_audit'):
    labels = json.loads(
        (V3_BUNDLES / role / 'v3_role_labels_manifest.json').read_text()
    )
    assert labels['status'] == 'ROLE_LABELS_MATERIALIZED_FROM_FROZEN_PARENT'
    assert labels['role_id'] == role
assert {path.name for path in (V3_BUNDLES / 'future_method_holdout').iterdir()} == future_files
print('Observed design/audit roles materialized; future role remains identity-only.')

In [ ]:
subprocess.run([
    EXPERIMENT_PYTHON,
    str(V5_REPO / 'scripts/build_rars_v5_pq_aware_100k_bundle.py'),
    '--design-role-dir', str(V3_BUNDLES / 'oracle_design'),
    '--audit-role-dir', str(V3_BUNDLES / 'oracle_audit'),
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--output-root', str(V5_BUNDLE),
    '--source-commit', V5_IMPLEMENTATION_COMMIT,
    '--protocol', str(V5_PROTOCOL_PATH),
    '--use-gpu',
], check=True, cwd=V5_REPO, env=EXPERIMENT_ENV)
pilot = json.loads((V5_BUNDLE / 'pilot_bundle_summary.json').read_text())
assert pilot['status'] == 'PILOT_BUNDLE_COMPLETE'
assert pilot['source_commit'] == V5_IMPLEMENTATION_COMMIT
assert pilot['pilot_docs'] == 100000
assert pilot['data_access']['future_method_holdout_opened'] is False
assert pilot['data_access']['external_collection_opened'] is False
assert pilot['data_access']['raw_qrels_opened_by_v5_builder'] is False
assert pilot['roles']['oracle_audit']['evaluation_candidates_are_label_independent'] is True
assert pilot['roles']['oracle_audit']['missing_positives_appended_to_pair_mining_candidates'] is False
print(json.dumps({
    'pilot_docs': pilot['pilot_docs'],
    'train_queries': pilot['roles']['oracle_design']['query_count'],
    'selection_queries': pilot['roles']['oracle_audit']['query_count'],
    'base_selection_recall_at_100': pilot['roles']['oracle_audit']['base_pq_recall_at_100'],
    'teacher_selection_recall_at_100': pilot['roles']['oracle_audit']['teacher_exact_recall_at_100'],
}, indent=2))

## Train and evaluate seed 42

This cell is the first v5 model-outcome access. It evaluates identity epoch 0, trains for eight frozen epochs, and reruns end-to-end hard-IVF-PQ selection retrieval after every epoch. Do not interrupt it or edit the durable output directory. A T4 run may take tens of minutes because every epoch re-encodes and searches the entire 100K pilot corpus.

In [ ]:
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
subprocess.run([
    EXPERIMENT_PYTHON,
    str(V5_REPO / 'scripts/train_rars_v5_pq_aware_adapter.py'),
    '--bundle-root', str(V5_BUNDLE),
    '--embeddings', str(CACHE / 'embeddings.fp16.memmap'),
    '--output-dir', str(OUTPUT),
    '--source-commit', V5_IMPLEMENTATION_COMMIT,
    '--protocol', str(V5_PROTOCOL_PATH),
], check=True, cwd=V5_REPO, env=EXPERIMENT_ENV)
print('Seed-42 pilot completed.')

In [ ]:
complete_path = OUTPUT / 'training_complete.json'
result_path = OUTPUT / 'pilot_result.json'
complete = json.loads(complete_path.read_text())
result = json.loads(result_path.read_text())
assert complete['status'] == 'TRAINING_COMPLETE'
assert result['status'] == 'TRAINING_COMPLETE'
assert complete['source_commit'] == V5_IMPLEMENTATION_COMMIT
assert complete['run_id'] == result['run_id']
assert complete['formal_decision'] == result['formal_decision']
verify_record(result_path, complete['result'])
verify_record(OUTPUT / 'training_started.json', complete['started'])
verify_record(OUTPUT / 'training_history.json', complete['history'])
for filename, record in complete['outputs'].items():
    verify_record(OUTPUT / filename, record)
assert result['formal_decision'] in {
    'GO_TO_THREE_SEED_100K_REPLICATION',
    'STOP_PQ_AWARE_100K_PILOT',
}
report = {
    'formal_decision': result['formal_decision'],
    'selected_epoch': result['selected_epoch'],
    'pair_count': result['pair_count'],
    'pq_induced_flip_pairs': result['pq_induced_flip_pairs'],
    'selection': result['selection'],
    'gates': result['gates'],
    'run_id': result['run_id'],
    'result_sha256': sha256_file(result_path),
}
print(json.dumps(report, indent=2, allow_nan=False))

## Checks and next steps

Interpret the formal decision literally:

- `STOP_PQ_AWARE_100K_PILOT`: at least one breadth, uncertainty, recovery, Top-10, or FP32 guardrail failed. Stop this configuration; do not rescue it by changing the loss after seeing the selection result.
- `GO_TO_THREE_SEED_100K_REPLICATION`: only exact seeds 43 and 44 replication may be designed next. This is not method success and does not authorize RARS training, a 1M rebuild, an external dataset, or the future holdout.

Return the printed report plus `pilot_result.json`, `training_complete.json`, and `training_history.json` for audit. The later adapter-plus-RARS experiment requires a separate preregistration and is attempted only after a replicated GO.